# Hardware-constrained parameter optimisation

This is a small, off-board recommendation tool. It **does not access or configure hardware**. Feasibility is checked before performance: the analogue chain must be non-damaging under a full-scale fault and linear at the requested measurement drive before CIC, FIR, FFT, or feedback settings are considered.

The model follows the real-coil wiring: `OUT1 -> pad/amplifier chain -> 0.3 pF -> tank -> 0.3 pF -> high-impedance IN1`. Power in dBm is used only at declared 50 ohm interfaces. An unterminated final amplifier is explicitly marked as a mismatch and receives the manufacturer's 20 dB no-load input derating.

## Editable hardware configuration

Edit only this cell to represent another bench setup. Amplifier values are conservative 40 MHz values from the Mini-Circuits data sheets and the coil-characterisation notebook. Measured gain overrides take precedence only when they are lower than the declared gain; safety limits always use the most conservative applicable value.

In [ ]:
from pathlib import Path
import json
import itertools
import time
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / 'fpga' / 'fir_coeffs.txt').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'fpga' / 'fir_coeffs.txt').exists():
    raise RuntimeError('Run this notebook from inside the Test_Project checkout')

# Fixed platform and coil arrangement
FS = 125e6
DAC_FULL_SCALE_DBM = 10.0       # +/-1 V peak into 50 ohm
ADC_FULL_SCALE_VPK = 1.0        # LV jumper
ADC_GUARD_FRAC = 0.80
C_COUPLE = 0.3e-12
F0 = 40.0e6
Q0 = 180.0
G_PASSIVE = 0.0158              # measured round-trip voltage transfer, no amplifier
NORMAL_DRIVE_FRAC = 0.0032
CABLE_LOSS_DB = 0.0
REQUIRED_NET_GAIN_DB = 30.5     # measured estimate for reaching K=1
DAMAGE_MARGIN_DB = 1.0
LINEAR_MARGIN_DB = 3.0
AVAILABLE_PADS_DB = [0, 5, 10, 15, 20, 25, 30]
TERMINATION_OPTIONS = [False, True]
TERMINATION_LOSS_DB = 6.0
MEASURED_GAIN_OVERRIDES_DB = {} # e.g. {'ZFL-500LN+': 28.1}
PI_FUTURE = {'enabled': False, 'Kp': None, 'Ki': None}

# FFT estimator experiment. dBFS means tone PEAK relative to ADC full-scale peak.
FFT_SNR_DB = np.arange(0, 61, 10)
FFT_LEVEL_DBFS = np.arange(-60, -9, 10)
FFT_FIXED_LEVEL_DBFS = -12.0
FFT_NOISE_FLOOR_DBFS = -70.0
FFT_LENGTHS = (1024, 4096)
FFT_BIN_OFFSETS = (0.0, 0.25, 0.5)
FFT_REPETITIONS = 6
FFT_ZERO_PAD_FACTOR = 4
FFT_TRACKED_BIN = 19.0
FFT_SEED = 2020

AMPLIFIERS = {
    'ZFL-500LN+': dict(gain_db=28.2, f_lo=0.1e6, f_hi=500e6,
                       max_input_dbm=5.0, p1db_dbm=7.9, open_linear_dbm=0.0,
                       noise_figure_db=2.9, supply_v=15.0),
    'ZFL-750+':   dict(gain_db=18.0, f_lo=0.2e6, f_hi=750e6,
                       max_input_dbm=5.0, p1db_dbm=9.0, open_linear_dbm=1.0,
                       noise_figure_db=6.0, supply_v=15.0),
}
ALLOWED_CHAINS = [(), ('ZFL-500LN+',), ('ZFL-750+',),
                  ('ZFL-500LN+', 'ZFL-750+')]

# Optional measured results from the coil notebook. Missing data is harmless.
measured = {}
results_path = ROOT / 'bench_coil_results.json'
if results_path.exists():
    measured = json.loads(results_path.read_text())
    F0 = float(measured.get('coil', {}).get('f0_hz', F0))
    Q0 = float(measured.get('coil', {}).get('q_swept', Q0))
    G_PASSIVE = float(measured.get('calibration', {}).get('g_passive', G_PASSIVE))
    print('Loaded measured coil values from', results_path)
else:
    print('Using editable defaults; bench_coil_results.json was not found')

# Complete acquisition-chain tuning. Structural alternatives are model-only.
ACQ_DECIMATIONS = (64, 128, 256, 512, 1024, 2048, 4096)
FIR_PASSBANDS = (0.10, 0.15, 0.20, 0.25)
FIR_TRANSITION_WIDTHS = (0.05, 0.08)
FIR_WINDOWS = ('blackman', 'hamming')
FIR_STRUCTURAL_TAPS = (127, 155, 255, 309, 511)
CIC_STRUCTURAL_ORDERS = (3, 4, 5)
DDS_STRUCTURAL_LUT_BITS = (12, 13, 14)
MIXER_STRUCTURAL_SHIFTS = (5, 6, 7)
DMA_STRUCTURAL_DEPTHS = (256, 512, 2048)
DMA_SAFE_BYTES_PER_S = 50e6
ACQ_MEMORY_LIMIT_BYTES = 16*1024*1024
ACQ_REFERENCE_SNR_DB = 20.0
ACQ_AVERAGES = (1, 4, 16)

linewidth_default = F0/Q0
stability_data = measured.get('stability', {})
feedback_data = measured.get('feedback', {})
drift_hz_s = abs(float(stability_data.get('drift_hz_per_min', 0.0)))/60.0
scatter_hz = abs(float(stability_data.get('scatter_hz', 0.0)))
shift_hz = abs(float(feedback_data.get('shift_max_hz', 0.0)))
PROFILE_CONFIG = {
    'fast tracked amplitude': dict(required_halfband_hz=max(5e3, 0.025*linewidth_default, drift_hz_s),
                                   max_latency_s=0.050, min_alias_db=50.0, max_ripple_db=0.15, min_stop_db=45.0, max_lo_tiles=1),
    'precise tracked amplitude': dict(required_halfband_hz=max(1e3, 5*scatter_hz),
                                      max_latency_s=0.500, min_alias_db=60.0, max_ripple_db=0.10, min_stop_db=55.0, max_lo_tiles=1),
    'wide resonance diagnostics': dict(required_halfband_hz=0.75*linewidth_default+shift_hz,
                                       max_latency_s=0.250, min_alias_db=45.0, max_ripple_db=1.0, min_stop_db=40.0, max_lo_tiles=4),
}

assert 0 < NORMAL_DRIVE_FRAC <= 1
assert 0 < ADC_GUARD_FRAC < 1
assert abs(C_COUPLE - 0.3e-12) < 1e-18, 'the protected bench arrangement uses 0.3 pF couplers'
assert not PI_FUTURE['enabled'], 'PI optimisation is deliberately deferred'
assert len(FFT_SNR_DB) and len(FFT_LEVEL_DBFS), 'FFT dB ranges must not be empty'
assert np.all(np.isfinite(FFT_SNR_DB)) and np.all(np.isfinite(FFT_LEVEL_DBFS))
assert np.max(FFT_LEVEL_DBFS) <= 0 and FFT_FIXED_LEVEL_DBFS <= 0, 'signal level cannot exceed 0 dBFS'
assert int(FFT_ZERO_PAD_FACTOR) == FFT_ZERO_PAD_FACTOR and FFT_ZERO_PAD_FACTOR >= 1
assert FFT_REPETITIONS >= 1 and all(int(n) == n and n >= 16 for n in FFT_LENGTHS)
assert all(R >= 64 and R <= 4096 and (R & (R-1)) == 0 for R in ACQ_DECIMATIONS)
assert all(0 < p < 0.5 for p in FIR_PASSBANDS)
assert all(v['required_halfband_hz'] > 0 and v['max_latency_s'] > 0 for v in PROFILE_CONFIG.values())

## Analogue-chain safety audit

For a matched interface, $P_{i+1}=P_i+G_i-A_i$ in dBm. Damage and linear operation are independent checks. A candidate must be linear at the normal drive. At full-scale fault level it may compress, but it must not exceed an absolute damage rating. The maximum safe and maximum linear DAC fractions are obtained by propagating every downstream limit back to OUT1.

The coupling reactance is $X_C=(2\pi f_0C)^{-1}$. The displayed capacitor current and power are conservative indicators, not a substitute for a measured coil dissipation limit.

In [ ]:
def dbm_from_fraction(frac):
    return DAC_FULL_SCALE_DBM + 20*np.log10(max(float(frac), 1e-15))

def fraction_from_dbm(dbm):
    return min(1.0, 10**((float(dbm)-DAC_FULL_SCALE_DBM)/20))

def amplifier_gain(name):
    declared = AMPLIFIERS[name]['gain_db']
    measured_gain = MEASURED_GAIN_OVERRIDES_DB.get(name, declared)
    return min(float(declared), float(measured_gain))

def audit_chain(chain, pad_db, terminated, drive_frac):
    p = dbm_from_fraction(drive_frac) - float(pad_db)
    cumulative = -float(pad_db)
    rows, safe_limits, linear_limits = [], [], []
    for index, name in enumerate(chain):
        amp = AMPLIFIERS[name]
        last = index == len(chain)-1
        output_matched = (not last) or bool(terminated)
        damage_limit = amp['max_input_dbm'] - (0.0 if output_matched else 20.0)
        gain = amplifier_gain(name)
        output = p + gain
        linear_limit = amp['p1db_dbm'] if output_matched else amp['open_linear_dbm']
        rows.append(dict(component=name, input_dbm=p, output_dbm=output,
                         damage_limit_dbm=damage_limit, linear_limit_dbm=linear_limit,
                         damage_margin_db=damage_limit-DAMAGE_MARGIN_DB-p,
                         linear_margin_db=linear_limit-LINEAR_MARGIN_DB-output,
                         matched=output_matched))
        safe_limits.append(damage_limit - DAMAGE_MARGIN_DB - cumulative)
        linear_limits.append(linear_limit - LINEAR_MARGIN_DB - (cumulative + gain))
        p, cumulative = output, cumulative + gain
    if terminated and chain:
        p -= TERMINATION_LOSS_DB
        cumulative -= TERMINATION_LOSS_DB
    p -= CABLE_LOSS_DB
    cumulative -= CABLE_LOSS_DB
    adc_fraction = drive_frac * G_PASSIVE * 10**(cumulative/20)
    adc_margin_db = 20*np.log10(ADC_GUARD_FRAC/max(adc_fraction, 1e-15))
    adc_dac_limit = DAC_FULL_SCALE_DBM + 20*np.log10(ADC_GUARD_FRAC/max(G_PASSIVE*10**(cumulative/20), 1e-15))
    safe_limits.append(adc_dac_limit)
    linear_limits.append(adc_dac_limit)
    max_safe_dbm = min([DAC_FULL_SCALE_DBM] + safe_limits)
    max_linear_dbm = min([DAC_FULL_SCALE_DBM] + safe_limits + linear_limits)
    return dict(rows=rows, final_dbm=p, net_gain_db=cumulative,
                adc_fraction=adc_fraction, adc_margin_db=adc_margin_db,
                max_safe_frac=fraction_from_dbm(max_safe_dbm),
                max_linear_frac=fraction_from_dbm(max_linear_dbm),
                damage_ok=all(r['damage_margin_db'] >= 0 for r in rows),
                linear_ok=all(r['linear_margin_db'] >= 0 for r in rows) and adc_margin_db >= 0)

def evaluate_candidate(chain, pad_db, terminated):
    if not chain and terminated:
        return None
    if any(not (AMPLIFIERS[n]['f_lo'] <= F0 <= AMPLIFIERS[n]['f_hi']) for n in chain):
        return None
    normal = audit_chain(chain, pad_db, terminated, NORMAL_DRIVE_FRAC)
    fault = audit_chain(chain, pad_db, terminated, 1.0)
    fault_damage_margin = min([r['damage_margin_db'] for r in fault['rows']] + [fault['adc_margin_db']])
    normal_linear_margin = min([r['linear_margin_db'] for r in normal['rows']] + [normal['adc_margin_db']])
    fault_safe = fault['damage_ok'] and fault['adc_margin_db'] >= 0
    feasible = fault_safe and normal['linear_ok'] and C_COUPLE <= 0.3e-12
    nf = sum(AMPLIFIERS[n]['noise_figure_db'] for n in chain)
    return dict(chain=chain, pad_db=float(pad_db), terminated=bool(terminated),
                normal=normal, fault=fault, fault_safe=fault_safe, feasible=feasible,
                fault_damage_margin_db=fault_damage_margin,
                normal_linear_margin_db=normal_linear_margin,
                reaches_k1=normal['net_gain_db'] >= REQUIRED_NET_GAIN_DB,
                noise_score_db=nf, components=len(chain)+(pad_db > 0)+bool(terminated))

candidates = [evaluate_candidate(chain, pad, term)
              for chain, pad, term in itertools.product(ALLOWED_CHAINS, AVAILABLE_PADS_DB, TERMINATION_OPTIONS)]
candidates = [c for c in candidates if c is not None]
feasible = [c for c in candidates if c['feasible']]
feasible.sort(key=lambda c: (-c['fault_damage_margin_db'], -int(c['reaches_k1']),
                             -c['normal']['adc_fraction'], c['components'], c['noise_score_db']))

print('FEASIBLE HARDWARE SHORTLIST (normal operation linear; full-scale fault non-damaging)')
print('chain                         pad term net/dB safe_frac linear_frac ADC/% fault_margin reaches_K1')
for c in feasible[:10]:
    label = '+'.join(c['chain']) or 'passive'
    print('%-29s %3.0f  %-4s %6.1f %9.5f %11.5f %5.2f %11.2f %s' %
          (label, c['pad_db'], str(c['terminated']), c['normal']['net_gain_db'],
           c['normal']['max_safe_frac'], c['normal']['max_linear_frac'],
           100*c['normal']['adc_fraction'], c['fault_damage_margin_db'], c['reaches_k1']))

def find_case(chain, pad, term):
    return next(c for c in candidates if c['chain'] == chain and c['pad_db'] == pad and c['terminated'] == term)

current = find_case(('ZFL-500LN+',), 0, False)
padded = find_case(('ZFL-500LN+',), 25, False)
proposed = find_case(('ZFL-500LN+', 'ZFL-750+'), 10, True)
assert not current['fault']['damage_ok']
assert padded['fault_damage_margin_db'] > current['fault_damage_margin_db']
assert padded['normal']['net_gain_db'] < current['normal']['net_gain_db']
assert proposed['normal']['net_gain_db'] == sum(amplifier_gain(n) for n in proposed['chain']) - 10 - TERMINATION_LOSS_DB - CABLE_LOSS_DB
assert audit_chain(('ZFL-500LN+',), 0, True, 1.0)['rows'][-1]['damage_limit_dbm'] - audit_chain(('ZFL-500LN+',), 0, False, 1.0)['rows'][-1]['damage_limit_dbm'] == 20.0
assert current['normal']['max_safe_frac'] <= 10**((-15-DAMAGE_MARGIN_DB-DAC_FULL_SCALE_DBM)/20) + 1e-12

XC = 1/(2*np.pi*F0*C_COUPLE)
print('\n0.3 pF coupling reactance at %.3f MHz: %.2f kohm' % (F0/1e6, XC/1e3))
print('PI placeholder:', PI_FUTURE)

## Inspect documented configurations

The table below deliberately shows failures. The current open-load/no-pad arrangement is unsafe under a full-scale feedback fault. Padding protects the amplifier but sacrifices loop gain. The documented two-amplifier, terminated arrangement restores gain, but its full-scale fault can still be compressed; the node audit distinguishes that survivable fault behaviour from normal linear operation.

In [ ]:
for title, case in [('current: LN, open, no pad', current),
                    ('conservative: LN, open, 25 dB pad', padded),
                    ('documented cascade: 10 dB pad, terminated', proposed)]:
    print('\n' + title)
    print('  feasible=%s, net=%+.1f dB, normal linear=%s, full-scale non-damaging=%s' %
          (case['feasible'], case['normal']['net_gain_db'], case['normal']['linear_ok'], case['fault_safe']))
    print('  component       input   output  damage-limit  linear-limit  fault damage margin')
    for normal_row, fault_row in zip(case['normal']['rows'], case['fault']['rows']):
        print('  %-14s %7.1f %8.1f %13.1f %13.1f %19.1f' %
              (normal_row['component'], normal_row['input_dbm'], normal_row['output_dbm'],
               normal_row['damage_limit_dbm'], normal_row['linear_limit_dbm'], fault_row['damage_margin_db']))
    if case['chain'] and not case['terminated']:
        print('  WARNING: final amplifier is not terminated; 50-ohm output power arithmetic is not physical.')

assert any(not c['feasible'] for c in candidates)
assert all(c['fault_safe'] and c['normal']['linear_ok'] for c in feasible)

## DDS, mixer, CIC and FIR models

The deployable model exactly follows the qualified RTL: 32-bit DDS accumulator, 12-bit sine-table address, 14x16-bit IQ mixer shifted by six bits, four-stage CIC with a 72-bit accumulator, 309-tap symmetric FIR with 18-bit Q1.17 coefficients, and a 256-word DMA FIFO. Alternatives with different widths or sizes are reported only as structural recommendations requiring a rebuild and timing qualification.

In [ ]:
FIR_Q = np.loadtxt(ROOT / 'fpga' / 'fir_coeffs.txt', comments='#', dtype=np.int64)
FIR_H = FIR_Q / float(1 << 17)
L, WIN = len(FIR_H), 24

def cic_exact(f, R, stages):
    f = np.asarray(f, dtype=float)
    denominator = R*np.sin(np.pi*f/FS)
    answer = np.ones_like(f)
    nz = np.abs(denominator) > 1e-30
    answer[nz] = np.abs(np.sin(np.pi*R*f[nz]/FS)/denominator[nz])**stages
    return answer

def response_from_coefficients(q, u, frac_bits=17):
    n = np.arange(len(q))-(len(q)-1)/2
    return np.abs(np.exp(-2j*np.pi*np.outer(u, n)) @ np.asarray(q))/float(1 << frac_bits)

def design_fir(ntaps, stages, passband, stopband, window_name, coef_bits=18):
    if ntaps < 3 or ntaps % 2 == 0: raise ValueError('FIR tap count must be odd and >=3')
    if not 0 < passband < stopband < 0.5: raise ValueError('require 0 < passband < stopband < 0.5')
    if coef_bits < 2 or coef_bits > 18: raise ValueError('coefficient width must be 2..18 bits')
    grid = 8192; ug = np.linspace(0, 0.5, grid//2+1); wanted = np.zeros_like(ug)
    droop = np.sinc(ug)**stages; wanted[ug <= passband] = 1/np.maximum(droop[ug <= passband], 1e-15)
    transition = (ug > passband) & (ug < stopband)
    x = (ug[transition]-passband)/(stopband-passband)
    wanted[transition] = (1/max(np.sinc(passband)**stages, 1e-15))*0.5*(1+np.cos(np.pi*x))
    full = np.roll(np.fft.irfft(wanted, n=grid), grid//2); half = ntaps//2
    h = full[grid//2-half:grid//2+half+1]
    window = np.blackman(ntaps) if window_name == 'blackman' else np.hamming(ntaps)
    h = h*window; h /= h.sum(); frac = coef_bits-1; q = np.round(h*(1 << frac)).astype(np.int64)
    q[half] += (1 << frac)-int(q.sum())
    if np.max(np.abs(q)) > (1 << (coef_bits-1))-1: raise ValueError('coefficient overflow')
    return q, frac

def fir_metrics(q, frac, stages, passband, stopband):
    u = np.linspace(0, 0.5, 601); hf = response_from_coefficients(q, u, frac)
    combined = np.sinc(u)**stages*hf; band = u <= passband; stop = u >= stopband
    ripple = 20*np.log10(max(combined[band].max()/max(combined[band].min(), 1e-15), 1))
    rejection = -20*np.log10(max(hf[stop].max(), 1e-15))
    boost = 20*np.log10(max(hf[band].max(), 1e-15))
    return dict(ripple_db=float(ripple), stop_db=float(rejection), peak_boost_db=float(boost), dc_gain=float(q.sum()/float(1 << frac)))

def cic_alias_rejection(R, stages, halfband_hz):
    fout = FS/R; band = min(float(halfband_hz), 0.49*fout)
    image = np.linspace(max(fout-band, 1), min(fout+band, FS/2), 201)
    first = np.max(cic_exact(image, R, stages))
    sum_alias = abs(((2*F0+FS/2) % FS)-FS/2)
    summed = float(cic_exact([sum_alias], R, stages)[0])
    return -20*np.log10(max(first, summed, 1e-15))

print('ACQUISITION PARAMETER DEPENDENCIES')
dependencies = [
 ('DDS increment/PACCW', 'LO quantisation and phase continuity'), ('DDS LUT bits', 'spurs and BRAM'),
 ('mixer widths/shift', 'DSP use, fractional resolution and headroom'), ('CIC R/order', 'rate, alias rejection, droop and width'),
 ('FIR bands/taps/window', 'flatness, rejection, delay and minimum R'), ('coefficient bits', 'quantisation, DSP width and DC gain'),
 ('record length', 'FFT resolution, memory and latency'), ('DMA depth/mode', 'burst tolerance and overwrite behaviour'),
 ('estimator/averaging', 'amplitude error and processing latency')]
for parameter, effect in dependencies: print('  %-24s %s' % (parameter, effect))

# Runtime-loadable coefficient candidates; no generated file is changed.
fir_designs = [dict(name='checked-in 309', q=FIR_Q, frac=17, taps=309, stages=4, passband=.20, stopband=.28, window='blackman')]
for pb, tw, window_name in itertools.product(FIR_PASSBANDS, FIR_TRANSITION_WIDTHS, FIR_WINDOWS):
    if pb+tw >= .5: continue
    q, frac = design_fir(309, 4, pb, pb+tw, window_name)
    fir_designs.append(dict(name='%s pb%.2f sb%.2f' % (window_name, pb, pb+tw), q=q, frac=frac, taps=309, stages=4, passband=pb, stopband=pb+tw, window=window_name))
for design in fir_designs: design.update(fir_metrics(design['q'], design['frac'], 4, design['passband'], design['stopband']))

digital_rows = []
for R in ACQ_DECIMATIONS:
    fout = FS/R; throughput = (L+1)/2+8 <= R
    digital_rows.append(dict(R=R, stages=4, fout=fout, width=24+4*12, delay=(L-1)/(2*fout),
                             throughput=throughput, feasible=bool(feasible) and (throughput or R < 256)))
print('\nR     fout/kHz Wacc FIR-active FIR-delay/ms DMA-MB/s')
for row in digital_rows:
    print('%4d %9.2f %4d %-10s %12.3f %8.3f' % (row['R'], row['fout']/1e3, row['width'], row['R'] >= 256, 1e3*row['delay'] if row['R'] >= 256 else 0, row['fout']*8/1e6))

# Structural comparisons are deliberately not deployable. Their nominal FIR response is evaluated.
structural_filter_cache = {}
for structural_stages, structural_taps in itertools.product(CIC_STRUCTURAL_ORDERS, FIR_STRUCTURAL_TAPS):
    structural_q, structural_frac = design_fir(structural_taps, structural_stages, .20, .28, 'blackman')
    structural_filter_cache[(structural_stages, structural_taps)] = fir_metrics(structural_q, structural_frac, structural_stages, .20, .28)
structural_rows = []
for stages, taps, lut_bits, mixer_shift, fifo_depth in itertools.product(CIC_STRUCTURAL_ORDERS, FIR_STRUCTURAL_TAPS, DDS_STRUCTURAL_LUT_BITS, MIXER_STRUCTURAL_SHIFTS, DMA_STRUCTURAL_DEPTHS):
    min_r = 1 << int(np.ceil(np.log2((taps+1)/2+8))); history_depth = 1 << int(np.ceil(np.log2(taps)))
    filter_metric = structural_filter_cache[(stages, taps)]
    structural_rows.append(dict(deployable=False, status='requires rebuild and timing qualification', stages=stages, taps=taps,
        lut_bits=lut_bits, mixer_shift=mixer_shift, fifo_depth=fifo_depth, wacc=24+stages*12, min_R=min_r,
        history_depth=history_depth, dds_spur_db=-(6.02*lut_bits-3.9), dds_bram=2**max(0, lut_bits-12),
        mixer_headroom_bits=mixer_shift-5, retained_fraction_bits=15-mixer_shift,
        fir_ripple_db=filter_metric['ripple_db'], fir_stop_db=filter_metric['stop_db']))
assert all(not r['deployable'] for r in structural_rows)
assert len(FIR_Q) == 309 and int(FIR_Q.sum()) == 1 << 17
assert abs(cic_exact([0], 4096, 4)[0]-1) < 1e-15 and 24+4*12 == 72
assert cic_alias_rejection(256, 4, 5e3) > cic_alias_rejection(256, 3, 5e3)
assert (L+1)/2+8 <= 256 and (L+1)/2+8 > 128
for bad in ((128,4,.2,.28,'blackman',18), (309,4,.3,.2,'blackman',18), (309,4,.2,.28,'blackman',19)):
    try: design_fir(bad[0], bad[1], bad[2], bad[3], bad[4], bad[5]); raise AssertionError('invalid FIR design accepted')
    except ValueError: pass

u = np.linspace(0, .5, 401); checked = fir_designs[0]; hf = response_from_coefficients(checked['q'], u, checked['frac'])
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for stages in CIC_STRUCTURAL_ORDERS: axes[0].plot(u, 20*np.log10(np.maximum(np.sinc(u)**stages, 1e-8)), label='order %d' % stages)
axes[0].set(xlabel='frequency / output rate', ylabel='amplitude [dB]', title='CIC order and droop', ylim=(-35,2)); axes[0].legend()
axes[1].scatter([d['ripple_db'] for d in fir_designs], [d['stop_db'] for d in fir_designs], c=[d['peak_boost_db'] for d in fir_designs], cmap='viridis')
axes[1].set(xlabel='combined passband ripple [dB]', ylabel='FIR stopband rejection [dB]', title='Runtime-loadable 309-tap designs')
for ax in axes: ax.grid(alpha=.3)
fig.tight_layout(); plt.show(); plt.close(fig)

## FFT estimator comparison

The tracked tone is downmixed and decimated in the FPGA, then measured in software. Four estimators receive the **same immutable record**: largest FFT bin, three-bin quadratic interpolation, four-times zero-padded FFT, and a sine/cosine least-squares projection at the tracked-frequency hint. The FFT methods all use the same Hann window, so this compares estimator technique rather than window choice.

Here dBFS is tone peak amplitude relative to ADC full-scale peak. SNR is tone RMS divided by injected-noise RMS. $\Delta f=f_{out}/N$ and $T_{record}=N/f_{out}$; processing time is measured separately.

In [ ]:
def _hann_spectrum(x, nfft=None):
    window = np.hanning(len(x))
    return np.abs(np.fft.rfft(x*window, n=nfft)), float(window.sum())

def estimate_peak_bin(x):
    spectrum, coherent_sum = _hann_spectrum(x)
    k = int(np.argmax(spectrum[1:]))+1
    return 2*spectrum[k]/coherent_sum

def estimate_quadratic(x):
    spectrum, coherent_sum = _hann_spectrum(x)
    k = int(np.argmax(spectrum[1:]))+1
    if not 1 <= k < len(spectrum)-1:
        return 2*spectrum[k]/coherent_sum
    left, centre, right = spectrum[k-1:k+2]
    curvature = left-2*centre+right
    delta = 0.0 if abs(curvature) < 1e-30 else 0.5*(left-right)/curvature
    return 2*(centre-0.25*(left-right)*delta)/coherent_sum

def estimate_zero_padded(x):
    nfft = int(FFT_ZERO_PAD_FACTOR)*len(x)
    spectrum, coherent_sum = _hann_spectrum(x, nfft=nfft)
    k = int(np.argmax(spectrum[1:]))+1
    return 2*spectrum[k]/coherent_sum

def estimate_tracked_projection(x):
    n = np.arange(len(x), dtype=float)
    omega = 2*np.pi*FFT_TRACKED_BIN/len(x)
    design = np.column_stack((np.cos(omega*n), np.sin(omega*n), np.ones(len(x))))
    coefficients = np.linalg.lstsq(design, x, rcond=None)[0]
    return float(np.hypot(coefficients[0], coefficients[1]))

ESTIMATORS = {
    'peak bin': estimate_peak_bin,
    'quadratic': estimate_quadratic,
    'zero-padded': estimate_zero_padded,
    'tracked LS': estimate_tracked_projection,
}

def make_tone_record(N, level_dbfs, offset, rng, snr_db=None, noise_floor_dbfs=None):
    amplitude = 10**(float(level_dbfs)/20)
    n = np.arange(N, dtype=float)
    x = amplitude*np.cos(2*np.pi*(FFT_TRACKED_BIN+offset)*n/N+0.37)
    if (snr_db is None) == (noise_floor_dbfs is None):
        raise ValueError('choose exactly one of snr_db or noise_floor_dbfs')
    tone_rms = amplitude/np.sqrt(2)
    noise_rms = tone_rms/10**(float(snr_db)/20) if snr_db is not None else 10**(float(noise_floor_dbfs)/20)/np.sqrt(2)
    return x+rng.normal(0.0, noise_rms, N), amplitude

def summarise(estimates, truth):
    relative = np.asarray(estimates)/truth-1.0
    return dict(bias=float(np.mean(relative)), rms=float(np.sqrt(np.mean(relative**2))),
                p95=float(np.percentile(np.abs(relative), 95)))

def run_fft_sweep(mode, values, seed=FFT_SEED):
    if mode not in ('snr', 'level') or len(values) == 0:
        raise ValueError('mode must be snr or level and values must not be empty')
    rng = np.random.default_rng(seed); rows = []
    decimations = sorted({d['R'] for d in digital_rows if d['feasible']})
    for R, N, offset, value in itertools.product(decimations, FFT_LENGTHS, FFT_BIN_OFFSETS, values):
        by_method = {name: [] for name in ESTIMATORS}
        for _ in range(FFT_REPETITIONS):
            kwargs = dict(snr_db=value, noise_floor_dbfs=None) if mode == 'snr' else dict(snr_db=None, noise_floor_dbfs=FFT_NOISE_FLOOR_DBFS)
            level = FFT_FIXED_LEVEL_DBFS if mode == 'snr' else value
            record, truth = make_tone_record(N, level, offset, rng, **kwargs)
            before = record.tobytes()
            for name, estimator in ESTIMATORS.items():
                by_method[name].append(estimator(record))
                assert record.tobytes() == before, 'an estimator modified the shared record'
        for name, estimates in by_method.items():
            summary = summarise(estimates, truth)
            rows.append(dict(mode=mode, value=float(value), R=R, N=N, offset=float(offset),
                             method=name, record_time=N*R/FS, **summary))
    return rows

# Noise-free known answer and expected off-bin behaviour.
known_rng = np.random.default_rng(1)
known, known_amp = make_tone_record(4096, -12, 0, known_rng, snr_db=300)
for estimator in ESTIMATORS.values():
    assert abs(estimator(known)/known_amp-1) < 2e-5
half, half_amp = make_tone_record(4096, -12, 0.5, np.random.default_rng(2), snr_db=300)
assert abs(estimate_peak_bin(half)/half_amp-1) > 0.10, 'half-bin scalloping should be visible'
assert abs(estimate_tracked_projection(half)/half_amp-1) > abs(estimate_tracked_projection(known)/known_amp-1)

snr_rows = run_fft_sweep('snr', FFT_SNR_DB)
level_rows = run_fft_sweep('level', FFT_LEVEL_DBFS)
repeat_check_a = run_fft_sweep('snr', FFT_SNR_DB[:1])
repeat_check_b = run_fft_sweep('snr', FFT_SNR_DB[:1])
assert repeat_check_a == repeat_check_b, 'seeded FFT sweep is not repeatable'
assert {r['value'] for r in snr_rows} == set(map(float, FFT_SNR_DB))
assert {r['value'] for r in level_rows} == set(map(float, FFT_LEVEL_DBFS))

# Processing cost is measured once per length and method, not inside the accuracy grid.
timing_rows = []
for N in FFT_LENGTHS:
    sample, _ = make_tone_record(N, FFT_FIXED_LEVEL_DBFS, 0.25, np.random.default_rng(9), snr_db=20)
    for name, estimator in ESTIMATORS.items():
        durations = []
        for _ in range(7):
            start = time.perf_counter(); estimator(sample); durations.append(time.perf_counter()-start)
        timing_rows.append(dict(N=N, method=name, processing_time=float(np.median(durations))))

print('METHOD SUMMARY (all configured SNRs, levels, offsets, lengths and feasible rates)')
print('method          bias/%    RMS/%    P95/%   median processing/us')
for name in ESTIMATORS:
    combined = [r for r in snr_rows+level_rows if r['method'] == name]
    timings = [r['processing_time'] for r in timing_rows if r['method'] == name]
    print('%-13s %8.3f %8.3f %8.3f %20.1f' %
          (name, 100*np.mean([r['bias'] for r in combined]),
           100*np.sqrt(np.mean([r['rms']**2 for r in combined])),
           100*np.percentile([r['p95'] for r in combined], 95), 1e6*np.median(timings)))
print('Record times span %.2f to %.2f ms; total update adds the method processing time.' %
      (1e3*min(r['record_time'] for r in snr_rows), 1e3*max(r['record_time'] for r in snr_rows)))

def aggregate(rows, method, value):
    selected = [r['rms'] for r in rows if r['method'] == method and r['value'] == float(value)]
    return float(np.sqrt(np.mean(np.square(selected))))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for name in ESTIMATORS:
    axes[0].semilogy(FFT_SNR_DB, [aggregate(snr_rows, name, v) for v in FFT_SNR_DB], 'o-', label=name)
    axes[1].semilogy(FFT_LEVEL_DBFS, [aggregate(level_rows, name, v) for v in FFT_LEVEL_DBFS], 'o-', label=name)
axes[0].set(xlabel='SNR [dB]', ylabel='relative amplitude RMS error', title='Fixed %.0f dBFS tone' % FFT_FIXED_LEVEL_DBFS)
axes[1].set(xlabel='tone level [dBFS peak]', ylabel='relative amplitude RMS error', title='Fixed %.0f dBFS noise floor' % FFT_NOISE_FLOOR_DBFS)
for ax in axes: ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.tight_layout(); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(7.5, 4))
x = np.arange(len(ESTIMATORS)); width = 0.35
for j, N in enumerate(FFT_LENGTHS):
    values = [1e6*next(r['processing_time'] for r in timing_rows if r['N'] == N and r['method'] == name) for name in ESTIMATORS]
    ax.bar(x+(j-(len(FFT_LENGTHS)-1)/2)*width, values, width, label='N=%d' % N)
ax.set(xticks=x, xticklabels=list(ESTIMATORS), ylabel='median processing time [us]', title='Estimator cost; record time is reported separately')
ax.legend(); ax.grid(axis='y', alpha=.3); fig.tight_layout(); plt.show(); plt.close(fig)

print('\nInterpretation: peak bin is the cheapest baseline; quadratic interpolation improves it')
print('with little extra work; zero-padding trades more FFT work for off-bin accuracy; tracked')
print('least-squares is efficient only while the LO/frequency hint remains sufficiently accurate.')

## End-to-end deployable profile search

Every candidate below uses the current qualified RTL. FIR alternatives change only the 155 writable coefficient words and are recommendations for a later controlled load. The search includes analogue safety, bandwidth, aliasing, FIR response and throughput, mixer/CIC widths, DMA rate and memory, FFT accuracy, averaging, and total latency.

In [ ]:
def estimator_metrics(N, method, R):
    effective_snr = ACQ_REFERENCE_SNR_DB+10*np.log10(R/64.0)
    target = min(FFT_SNR_DB, key=lambda v: abs(float(v)-effective_snr))
    rows = [r for r in snr_rows if r['N'] == N and r['method'] == method and r['value'] == float(target)]
    processing = next(r['processing_time'] for r in timing_rows if r['N'] == N and r['method'] == method)
    return float(np.sqrt(np.mean([r['rms']**2 for r in rows]))), float(np.percentile([r['p95'] for r in rows], 95)), processing

def evaluate_acquisition(profile_name, config, R, N, method, averages, fir_design):
    fout = FS/R; required = config['required_halfband_hz']; active = fir_design is not None
    usable_halfband = fir_design['passband']*fout if active else 0.10*fout
    lo_tiles = max(1, int(np.ceil(required/usable_halfband)))
    tile_halfband = required/lo_tiles
    alias_db = cic_alias_rejection(R, 4, tile_halfband)
    if active:
        ripple_db, stop_db = fir_design['ripple_db'], fir_design['stop_db']
        fir_delay = (fir_design['taps']-1)/(2*fout)
        throughput = (fir_design['taps']+1)/2+8 <= R
        coefficient_ok = np.max(np.abs(fir_design['q'])) <= (1 << 17)-1 and abs(fir_design['dc_gain']-1) < 1e-12
        history_ok = fir_design['taps'] <= 512
    else:
        edge_gain = float(cic_exact([tile_halfband], R, 4)[0]) if tile_halfband < fout/2 else 0.0
        ripple_db = -20*np.log10(max(edge_gain, 1e-15)); stop_db = np.inf
        fir_delay = 0.0; throughput = True; coefficient_ok = True; history_ok = True
    error, p95, processing = estimator_metrics(N, method, R)
    record_time = N/fout; memory_bytes = N*8*averages*lo_tiles; byte_rate = fout*8
    pipeline_time = (1+2+6+(163 if active else 0)+8)/FS
    total_latency = lo_tiles*(averages*(record_time+processing)+fir_delay+pipeline_time)
    failures = []
    checks = [
      (bool(feasible), 'analogue safety/headroom'), (lo_tiles <= config['max_lo_tiles'], 'required bandwidth/LO tiles'),
      (alias_db >= config['min_alias_db'], 'CIC alias rejection'), (ripple_db <= config['max_ripple_db'], 'passband ripple'),
      ((not active) or stop_db >= config['min_stop_db'], 'FIR stopband rejection'), (throughput, 'FIR throughput'),
      (coefficient_ok, 'FIR coefficient/DC gain'), (history_ok, 'FIR history depth'), (60 >= 33+18+int(np.ceil(np.log2(155))), 'FIR accumulator width'),
      (24+4*12 <= 72, 'CIC accumulator width'), (6-5 >= 1, 'mixer headroom'),
      (byte_rate <= DMA_SAFE_BYTES_PER_S, 'DMA bandwidth'), (memory_bytes <= ACQ_MEMORY_LIMIT_BYTES, 'capture memory'),
      (total_latency <= config['max_latency_s'], 'maximum latency')]
    failures = [label for ok, label in checks if not ok]
    return dict(profile=profile_name, deployable=True, R=R, dec_log2=int(np.log2(R)), fout=fout, N=N, method=method,
                averages=averages, fir_name='bypass' if not active else fir_design['name'], fir= fir_design,
                bypass=not active, usable_halfband=usable_halfband, lo_tiles=lo_tiles, tile_halfband=tile_halfband, alias_db=alias_db, ripple_db=ripple_db, stop_db=stop_db,
                error=error/np.sqrt(averages), p95=p95/np.sqrt(averages), processing=processing,
                record_time=record_time, fir_delay=fir_delay, total_latency=total_latency,
                byte_rate=byte_rate, memory_bytes=memory_bytes, resolution_hz=fout/N, cic_noise_gain_db=10*np.log10(R/64.0), failures=failures, feasible=not failures)

acquisition_candidates = []
for profile_name, config in PROFILE_CONFIG.items():
    for R, N, method, averages in itertools.product(ACQ_DECIMATIONS, FFT_LENGTHS, ESTIMATORS, ACQ_AVERAGES):
        choices = fir_designs if R >= 256 else [None]
        for fir_design in choices:
            acquisition_candidates.append(evaluate_acquisition(profile_name, config, R, N, method, averages, fir_design))

def dominates(a, b, keys):
    av = [a[k] for k in keys]; bv = [b[k] for k in keys]
    return all(x <= y for x, y in zip(av, bv)) and any(x < y for x, y in zip(av, bv))

def pareto(rows, keys):
    return [r for r in rows if not any(dominates(other, r, keys) for other in rows if other is not r)]

objective_keys = {
 'fast tracked amplitude': ('total_latency', 'error', 'memory_bytes'),
 'precise tracked amplitude': ('error', 'p95', 'total_latency'),
 'wide resonance diagnostics': ('resolution_hz', 'error', 'total_latency'),
}
selected_profiles, pareto_profiles = {}, {}
for profile_name in PROFILE_CONFIG:
    good = [r for r in acquisition_candidates if r['profile'] == profile_name and r['feasible']]
    if not good: raise RuntimeError('no feasible acquisition setting for '+profile_name)
    front = pareto(good, objective_keys[profile_name]); pareto_profiles[profile_name] = front
    if profile_name == 'fast tracked amplitude': key = lambda r: (r['total_latency'], r['error'], r['memory_bytes'])
    elif profile_name == 'precise tracked amplitude': key = lambda r: (r['error'], r['p95'], -r['alias_db'], r['total_latency'])
    else: key = lambda r: (r['resolution_hz'], -r['alias_db'], r['total_latency'], r['memory_bytes'])
    selected_profiles[profile_name] = min(front, key=key)

lo_finc = int(round(F0/FS*(1 << 32))) & 0xffffffff
lo_actual_hz = lo_finc*FS/float(1 << 32)
print('QUALIFIED DEPLOYABLE STAGE METRICS')
print('  DDS: 32-bit increment, error %+.6f Hz; LUT12 estimated worst spur %.1f dBc; 1 BRAM; 1 clock' % (lo_actual_hz-F0, -(6.02*12-3.9)))
print('  mixer: 14x16 -> 30-bit product, shift 6 -> 24 bits; 9 fractional bits retained; 1 headroom bit; 2 DSPs; 2 clocks')
print('  CIC: order 4, WACC 72, 32-bit output with 8 fractional bits; 6-clock comb/output pipeline')
print('  FIR: 309 taps, 155 MACs, Q1.17 coefficients, 60-bit accumulator, 512-sample history')
print('  DMA: 64-bit I/Q words, FIFO depth 256; qualified timing WNS +0.010 ns')
print('DEPLOYABLE ACQUISITION PROFILES (recommendations only)')
for profile_name, row in selected_profiles.items():
    print('\n'+profile_name.upper())
    print('  R=%d (DEC_LOG2=%d), fout=%.2f kHz, N=%d, %s, average=%d' % (row['R'], row['dec_log2'], row['fout']/1e3, row['N'], row['method'], row['averages']))
    print('  FIR=%s; usable half-band %.2f kHz x %d LO tile(s); ripple %.3f dB; alias %.1f dB' % (row['fir_name'], row['usable_halfband']/1e3, row['lo_tiles'], row['ripple_db'], row['alias_db']))
    print('  error %.3f%%, P95 %.3f%%, resolution %.3f Hz, latency %.2f ms, memory %.1f KiB, CIC noise gain %.1f dB' % (100*row['error'], 100*row['p95'], row['resolution_hz'], 1e3*row['total_latency'], row['memory_bytes']/1024, row['cic_noise_gain_db']))
    print('  registers: LO_FINC=0x%08X DEC_LOG2=%d CTRL.corrector_bypass=%d; coefficient set=%s' % (lo_finc, row['dec_log2'], row['bypass'], row['fir_name']))
    print('  Pareto candidates retained:', len(pareto_profiles[profile_name]))

failure_counts = {}
for row in acquisition_candidates:
    for failure in row['failures']: failure_counts[failure] = failure_counts.get(failure, 0)+1
print('\nFAILED-CONSTRAINT AUDIT')
for name, count in sorted(failure_counts.items(), key=lambda item: (-item[1], item[0])): print('  %-28s %5d candidates' % (name, count))

structural_shortlist = [r for r in structural_rows if r['mixer_headroom_bits'] >= 1 and r['min_R'] <= 256 and r['dds_spur_db'] <= -68 and r['fir_ripple_db'] <= .15 and r['fir_stop_db'] >= 45]
structural_shortlist.sort(key=lambda r: (r['min_R'], r['taps'], -r['stages'], r['dds_bram'], -r['fifo_depth']))
print('\nSTRUCTURAL IDEAS -- NOT DEPLOYABLE')
for row in structural_shortlist[:6]:
    print('  CIC%d WACC%d, FIR%d ripple %.3f dB stop %.1f dB minR%d, DDS%d (%+.1f dBc), shift%d, FIFO%d -- %s' %
          (row['stages'], row['wacc'], row['taps'], row['fir_ripple_db'], row['fir_stop_db'], row['min_R'], row['lut_bits'], row['dds_spur_db'], row['mixer_shift'], row['fifo_depth'], row['status']))

assert all(r['deployable'] and r['feasible'] for r in selected_profiles.values())
assert all(r['total_latency'] <= PROFILE_CONFIG[name]['max_latency_s'] for name, r in selected_profiles.items())
assert all(not r['deployable'] for r in structural_shortlist)
assert all(abs(r['byte_rate']-r['fout']*8) < 1e-12 and r['memory_bytes'] == r['N']*8*r['averages']*r['lo_tiles'] for r in acquisition_candidates)
assert all(abs(r['record_time']-r['N']/r['fout']) < 1e-15 for r in acquisition_candidates)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
for ax, (profile_name, row) in zip(axes, selected_profiles.items()):
    uu = np.linspace(0, .5, 401); cic = np.sinc(uu)**4
    total = cic if row['fir'] is None else cic*response_from_coefficients(row['fir']['q'], uu, row['fir']['frac'])
    ax.plot(uu*row['fout']/1e3, 20*np.log10(np.maximum(total, 1e-8)))
    ax.axvline(row['tile_halfband']/1e3, color='C1', ls='--', label='required per LO tile')
    ax.set(title=profile_name.replace(' amplitude',''), xlabel='baseband frequency [kHz]', ylabel='response [dB]', ylim=(-80,3)); ax.grid(alpha=.3); ax.legend(fontsize=7)
fig.suptitle('End-to-end CIC/FIR response of deployable profiles'); fig.tight_layout(); plt.show(); plt.close(fig)

## Interpretation and commissioning boundary

The shortlist is a recommendation, not permission to energise hardware. A fixed pad, correct 50 ohm termination, DC blocks, amplifier power sequencing, and separated drive/pickup cables are physical safeguards that software cannot replace. Before use, measure each attenuator, verify low-level chain gain into a dummy load, and confirm OUT1 full scale. Cryogenic-amplifier and electrode configurations must not be added until their maximum safe input levels are known.

First-order feedback measurement and interpretation are deliberately separate from this acquisition-chain study. Use resonator_coil_characterisation.ipynb for the guided Red Pitaya measurement, automatic analysis, and separate maximum-Q and maximum-shift reports.

To rerun this notebook for another analogue setup, edit the hardware configuration cell, restart the kernel, and run all cells. PI parameters remain intentionally disabled until an electron can be trapped and the amplitude plant is measurable.